In [ ]:
%matplotlib ipympl

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import casadi as ca

## limits

In [ ]:
dt = 1.0 / 200.0

In [ ]:
# vectors
bots = np.array(
    [
        [952.5055, 91.0723, -1410.0000],
        [-398.5396, 869.5826, -1409.8621],
        [-555.4801, 779.1038, -1410.0000],
        [-555.0219, -779.3507, -1409.6010],
        [-398.5396, -869.9006, -1410.0000],
        [952.7381, -89.7865, -1409.8718],
    ]
)
bots *= 1e-3
tops = np.array(
    [
        [314.4868, 327.8608, -111.0000],
        [126.7447, 436.2739, -111.1102],
        [-441.2953, 107.9497, -111.0000],
        [-441.2826, -108.6562, -111.3975],
        [126.7447, -436.2688, -111.0000],
        [314.5916, -328.3827, -110.9675],
    ]
)
tops *= 1e-3

top_normals = np.array(
    [
        [0.435014, -0.162005, -0.885729],
        [-0.357803, 0.295803, -0.885708],
        [-0.077200, 0.457799, -0.885698],
        [-0.077200, -0.457799, -0.885698],
        [-0.357803, -0.295803, -0.885708],
        [0.435014, 0.162005, -0.885729],
    ]
)
top_normals /= np.linalg.norm(top_normals, axis=1)[:, np.newaxis]
bot_normals = np.array(
    [
        [-0.435014, 0.162005, 0.885729],
        [0.357803, -0.295803, 0.885708],
        [0.077200, -0.457799, 0.885698],
        [0.077200, 0.457799, 0.885698],
        [0.357803, 0.295803, 0.885708],
        [-0.435014, -0.162005, 0.885729],
    ]
)
bot_normals /= np.linalg.norm(bot_normals, axis=1)[:, np.newaxis]

In [ ]:
# limits

top_max_angle = 42.0 * np.pi / 180.0
bot_max_angle = 42.0 * np.pi / 180.0

max_roll = 35.0 * np.pi / 180.0
max_pitch = 35.0 * np.pi / 180.0
max_yaw = 35.0 * np.pi / 180.0

min_leg_pos = 1160.410000 * 1e-3
max_leg_pos = 1770.010000 * 1e-3

max_leg_vel = 20.0 / 39.37

max_cart_table_acc = 8.0

max_cart_vel = 10.0
max_cart_acc = 18.0
max_angle_vel = 4.8
max_angle_acc = 2100.0

human_displacement = np.array([0.0, 0.0, 0.588])

In [ ]:
np.mean([min_leg_pos, max_leg_pos])

In [ ]:
# mid point at default rest
np.mean(np.linalg.norm(tops - bots, axis=1))

## rotation conventions

In [ ]:
t = sp.Symbol('t')
phi = sp.Function(r'\phi')(t)  # type: ignore
theta = sp.Function(r'\theta')(t)  # type: ignore
psi = sp.Function(r'\psi')(t)  # type: ignore

In [ ]:
def make_R() -> sp.Matrix:
    R_x = sp.Matrix(
        [
            [1, 0, 0],
            [0, sp.cos(phi), -sp.sin(phi)],
            [0, sp.sin(phi), sp.cos(phi)],
        ]
    )  # roll
    R_y = sp.Matrix(
        [
            [sp.cos(theta), 0, sp.sin(theta)],
            [0, 1, 0],
            [-sp.sin(theta), 0, sp.cos(theta)],
        ]
    )  # pitch
    R_z = sp.Matrix(
        [
            [sp.cos(psi), -sp.sin(psi), 0],
            [sp.sin(psi), sp.cos(psi), 0],
            [0, 0, 1],
        ]
    )  # yaw
    return R_z * R_y * R_x  # type: ignore

R = make_R()
R

Suppose that we have the rigid transformation
\begin{equation*}
  x_{\mathrm{f}} = R \, x_{\mathrm{h}} + \Delta_{\mathrm{h}}
\end{equation*}
with $x_{\mathrm{f}}$ the fixed frame and $x_{\mathrm{h}}$ the (moving) head frame.
Namely, the rotation $R = R(t)$ and the translation $\Delta = \Delta(t)$ are functions of time.
Decompose
\begin{align*}
  R(t) &=: R(0) \, R_{\mathrm{h}}(t) =: R_{\mathrm{r}}(t) \, R_{\mathrm{h}}(t) \\
  \Delta(t) &=: \Delta(0) + R(0) \, \Delta_{\mathrm{h}}(t) =: \Delta_{\mathrm{r}}(t) + R_{\mathrm{r}}(t) \, \Delta_{\mathrm{h}}(t).
\end{align*}
Namely, $\dot{R}_{\mathrm{r}} = 0$ and $\dot{\Delta}_{\mathrm{r}} = 0$.
The idea is that the head frame, at each instant, is evolving from the previous (instantaneous) pose in space.
This is a better model for angular velocity since, e.g., the table may be tilted while matching a desired $x\text{-acceleration}$.
Then
\begin{align*}
  x_{\mathrm{r}} &= R_{\mathrm{h}} \, x_{\mathrm{h}} + \Delta_{\mathrm{h}} \\
  \implies \quad \dot{x}_{\mathrm{r}} &= \dot{R}_{\mathrm{h}} \, x_{\mathrm{h}} + \dot{\Delta}_{\mathrm{h}} \\
  \implies \quad \dot{x}_{\mathrm{r}} &= \dot{R}_{\mathrm{h}} \, R^{\operatorname{T}}_{\mathrm{h}}(x_h - \Delta_h) + \dot{\Delta}_{\mathrm{h}}
\end{align*}
with $R_{\mathrm{h}}(t) = R(0)^{\operatorname{T}} \, R(t)$.
So, at $t = 0$, we have $\dot{R}_{\mathrm{h}} = R^{\operatorname{T}} \, \dot{R}$ and $R_{\mathrm{h}} = R^{\operatorname{T}} \, R = I$ so that
\begin{equation*}
  \dot{x}_{\mathrm{r}} = R^{\operatorname{T}} \, \dot{R} \, (x_{\mathrm{h}} - \Delta_{\mathrm{h}}) + \dot{\Delta_{\mathrm{h}}}.
\end{equation*}
This means that we have the angular velocity matrix $R^{\operatorname{T}} \, \dot{R}$, which is not the angular velocity matrix of the table (with respect to the fixed world frame), which is given by $\dot{R} \, R^{\operatorname{T}}$.
(Apparently, these two are quite different from each other.)

Recall that if we have a vector $\omega = (\omega_x, \omega_y, \omega_z)$, the corresponding skew symmetric matrix follows the sign conventions
\begin{equation*}
  [\omega]_\times = \begin{bmatrix} 0 & -\omega_z & \omega_y \\ \omega_z & 0 & -\omega_x \\ -\omega_y & \omega_x & 0 \end{bmatrix}.
\end{equation*}
This tells us that the (zero based) indices of interest are $(2, 1) \mapsto \omega_x$, $(0, 2) \mapsto \omega_y$, and $(1, 0) \mapsto \omega_z$.

In [ ]:
def make_omega_vec(world=False):
    """Map euler angle derivatives to head frame angular velocity."""
    if world:
        # table angular velocity (wrt world)
        omega_mat = sp.simplify(R.diff(t) * R.T)
    else:
        # head angular velocity (wrt instantaneous table)
        omega_mat = sp.simplify(R.T * R.diff(t))

    omega_vec = sp.Matrix([omega_mat[2, 1], omega_mat[0, 2], omega_mat[1, 0]])
    omega_vec = sp.simplify(omega_vec)
    
    return omega_vec

def make_PHI(world=False):
    """Map euler angle derivatives to head frame angular velocity."""
    if world:
        # table angular velocity (wrt world)
        omega_mat = sp.simplify(R.diff(t) * R.T)
    else:
        # head angular velocity (wrt instantaneous table)
        omega_mat = sp.simplify(R.T * R.diff(t))

    omega_vec = sp.Matrix([omega_mat[2, 1], omega_mat[0, 2], omega_mat[1, 0]])
    omega_vec = sp.simplify(omega_vec)

    euler_diff = [phi.diff(t), theta.diff(t), psi.diff(t)]
    PHI = sp.Matrix(3, 3, lambda i, j: omega_vec[i].coeff(euler_diff[j]))
    PHI = sp.simplify(PHI)
    
    return PHI

PHI = make_PHI()
PHI

In [ ]:
eval_vars = {
    phi: sp.Symbol("phi"),
    theta: sp.Symbol("theta"),
    psi: sp.Symbol("psi"),
}

res_human = sp.lambdify(
    list(eval_vars.values()),
    make_PHI(world=False).subs(eval_vars),
    modules="jax",
).__doc__
print(res_human.replace("array(", "jnp.array(").replace("sin(", "jnp.sin(").replace("cos(", "jnp.cos("))

res_world = sp.lambdify(
    list(eval_vars.values()),
    make_PHI(world=True).subs(eval_vars),
    modules="jax",
).__doc__
print(res_world.replace("array(", "jnp.array(").replace("sin(", "jnp.sin(").replace("cos(", "jnp.cos("))

In [ ]:
omega_r = make_omega_vec(world=True)
omega_h = make_omega_vec(world=False)
sp.simplify(omega_r.dot(omega_r) - omega_h.dot(omega_h))

In [ ]:
omega_dot_r = omega_r.diff(t)
omega_dot_h = omega_h.diff(t)
sp.simplify(omega_dot_r.dot(omega_dot_r) - omega_dot_h.dot(omega_dot_h))

In [ ]:
# not as nice...
# omega_dot_dot_r = omega_dot_r.diff(t)
# omega_dot_dot_h = omega_dot_h.diff(t)
# sp.simplify(omega_dot_dot_r.dot(omega_dot_dot_r) - omega_dot_dot_h.dot(omega_dot_dot_h))

## inverse kinematics

**_Remark._**

We may want to use, e.g., `sp.Symbol(R, commutative=False)` when initially computing fancy quantities with matrices, before expanding.
This might yield nicer final expressions.
But this would be subtle, unless I write new non-commutative symbol classes (or find an open source implementation in the enigma that is GitHub).
Cf. https://stackoverflow.com/questions/76281962/in-sympy-is-it-possible-to-create-a-matrix-symbol-that-is-a-function.


In [ ]:
x = sp.Matrix([sp.Function("x")(t), sp.Function("y")(t), sp.Function("z")(t)])  # type: ignore

In [ ]:
leg_vars = [sp.Function(f"\\ell_{i}")(t) for i in range(6)]  # type: ignore

top_vars = [
    sp.Matrix([
        sp.Symbol(f"t_{{{i},x}}"),
        sp.Symbol(f"t_{{{i},y}}"),
        sp.Symbol(f"t_{{{i},z}}"),
    ])
    for i in range(6)
]
bot_vars = [
    sp.Matrix([
        sp.Symbol(f"b_{{{i},x}}"),
        sp.Symbol(f"b_{{{i},y}}"),
        sp.Symbol(f"b_{{{i},z}}"),
    ])
    for i in range(6)
]

# top_vars = [sp.MatrixSymbol(f"t_{i}", 3, 1) for i in range(6)]
# bot_vars = [sp.MatrixSymbol(f"b_{i}", 3, 1) for i in range(6)]

In [ ]:
def get_lengths():
    lengths = []
    for i in range(6):
        top_i = R @ top_vars[i] + x
        diff_i = top_i - bot_vars[i]
        ell_i = sp.sqrt((diff_i.T * diff_i)[0, 0])  # get scalar
        lengths.append(ell_i)
    lengths = sp.Matrix(lengths)
    lengths = sp.simplify(lengths)
    return lengths

lengths = get_lengths()
lengths

In [ ]:
# these are not nice, and consequently, not really useful...
# (we should use more interesting formulas derived by hand)
lengths_squared = lengths.applyfunc(lambda ell: ell**2)
lengths_squared = sp.simplify(lengths_squared)
lengths_squared

## joint angle limits

In [ ]:
top_n_vars = [
    sp.Matrix(
        [
            sp.Symbol(f"t_{{n,{i},x}}"),
            sp.Symbol(f"t_{{n,{i},y}}"),
            sp.Symbol(f"t_{{n,{i},z}}"),
        ]
    )
    for i in range(6)
]
bot_n_vars = [
    sp.Matrix(
        [
            sp.Symbol(f"b_{{n,{i},x}}"),
            sp.Symbol(f"b_{{n,{i},y}}"),
            sp.Symbol(f"b_{{n,{i},z}}"),
        ]
    )
    for i in range(6)
]

In [ ]:
def cross_product(a, b):
    """Cross product of two vectors."""
    return sp.Matrix(
        [
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        ]
    )

In [ ]:
def get_angle_limits():
    top_angles = []
    bot_angles = []
    for i in range(6):
        top_i = R @ top_vars[i] + x
        bot_i = bot_vars[i]
        top_cross_i = cross_product(top_n_vars[i], (top_i - bot_i).normalized())
        bot_cross_i = cross_product(bot_n_vars[i], (top_i - bot_i).normalized())
        top_angle_i = sp.asin(sp.sqrt(top_cross_i.dot(top_cross_i)))
        bot_angle_i = sp.asin(sp.sqrt(bot_cross_i.dot(bot_cross_i)))
        top_angles.append(top_angle_i)
        bot_angles.append(bot_angle_i)
    return top_angles, bot_angles

top_angles, bot_angles = get_angle_limits()
top_angles

In [ ]:
# takes about 40 seconds to run
# sp.simplify(top_angles[0])

## human displacement

We control the robot rotation and translation, in Cartesian coordinates, but we also have the human displaced above the table.
\begin{align*}
  x_{\mathrm{f}} = R_{\mathrm{r}} \, x_{\mathrm{r}} + \Delta_{\mathrm{r}} \\
  x_{\mathrm{r}} = x_{\mathrm{h}} + \Delta_{\mathrm{h}},
\end{align*}
with $\dot{x}_{\mathrm{h}} = \dot{\Delta}_{\mathrm{h}} = 0$.
Combining these yields
\begin{equation*}
  x_{\mathrm{f}} = R_{\mathrm{r}} \, x_{\mathrm{h}} + [R_{\mathrm{r}} \, \Delta_{\mathrm{h}} + \Delta_{\mathrm{r}}],
\end{equation*}
so that we have a head frame angular acceleration of
\begin{equation*}
  a_{\mathrm{h}} = \ddot{R}_{\mathrm{r}} \, \Delta_{\mathrm{h}} + \ddot{\Delta}_{\mathrm{r}}.
\end{equation*}

In [ ]:
# for casadi, we need an expression for \ddot{R}, because this is relatively nontrivial
R_ddot = sp.simplify(R.diff(t, 2))
eval_vars = {
    phi: sp.Symbol("phi"),
    theta: sp.Symbol("theta"),
    psi: sp.Symbol("psi"),
    phi.diff(t): sp.Symbol("phi_dot"),
    theta.diff(t): sp.Symbol("theta_dot"),
    psi.diff(t): sp.Symbol("psi_dot"),
    phi.diff(t, 2): sp.Symbol("phi_ddot"),
    theta.diff(t, 2): sp.Symbol("theta_ddot"),
    psi.diff(t, 2): sp.Symbol("psi_ddot"),
}
R_ddot_eval = sp.lambdify(list(eval_vars.values()), R_ddot.subs(eval_vars), modules="numpy")

# coerce the printing to behave nicely with casadi, without hand editing
print(R_ddot_eval.__doc__.replace("sin(", "ca.sin(").replace("cos(", "ca.cos(").replace("array", ""))

## numerical testing

In [ ]:
# t_eval = (0.0, 0.0, 0.0)
# t_dot_eval = (1.0, -1.0, 1.0)
# t_ddot_eval = (-2.0, 2.0, -2.0)

t_eval = tuple(map(lambda x: float(x), np.random.uniform(-np.pi, np.pi, 3)))
t_dot_eval = tuple(map(lambda x: float(x), np.random.uniform(-np.pi, np.pi, 3)))
t_ddot_eval = tuple(map(lambda x: float(x), np.random.uniform(-np.pi, np.pi, 3)))
t_eval, t_dot_eval, t_ddot_eval

In [ ]:
eval_vars_R_dot = eval_vars.copy()
del eval_vars_R_dot[phi.diff(t, 2)]
del eval_vars_R_dot[theta.diff(t, 2)]
del eval_vars_R_dot[psi.diff(t, 2)]
R_dot = sp.simplify(R.diff(t))
R_dot_eval = sp.lambdify(list(eval_vars_R_dot.values()), R_dot.subs(eval_vars_R_dot), modules="numpy")

# conforms with a jax.jvp computation, performed below
R_dot_eval(*t_eval, *t_dot_eval)

In [ ]:
R_ddot_eval(*t_eval, *t_dot_eval, *t_ddot_eval)

In [ ]:
import jax
import jax.numpy as jnp

In [ ]:
@jax.jit
def _get_R(phi: float, theta: float, psi: float) -> jax.Array:
    """Get the rotation matrix."""
    R_x = jnp.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, jnp.cos(phi), -jnp.sin(phi)],
            [0.0, jnp.sin(phi), jnp.cos(phi)],
        ]
    )  # roll
    R_y = jnp.array(
        [
            [jnp.cos(theta), 0.0, jnp.sin(theta)],
            [0.0, 1.0, 0.0],
            [-jnp.sin(theta), 0.0, jnp.cos(theta)],
        ]
    )  # pitch
    R_z = jnp.array(
        [
            [jnp.cos(psi), -jnp.sin(psi), 0.0],
            [jnp.sin(psi), jnp.cos(psi), 0.0],
            [0.0, 0.0, 1.0],
        ]
    )  # yaw
    return R_z @ R_y @ R_x

In [ ]:
jax.jvp(_get_R, t_eval, t_dot_eval)[1]

In [ ]:
_get_R_dot_0 = lambda t_eval: jax.jvp(_get_R, t_eval, t_dot_eval)[1]  # noqa: E731
_get_R_dot_1 = lambda t_dot_eval: jax.jvp(_get_R, t_eval, t_dot_eval)[1]  # noqa: E731

jax.jvp(_get_R_dot_0, (t_eval,), (t_dot_eval,))[1] + jax.jvp(_get_R_dot_1, (t_dot_eval,), (t_ddot_eval,))[1]

In [ ]:
np.linalg.norm(R_dot_eval(*t_eval, *t_dot_eval) - jax.jvp(_get_R, t_eval, t_dot_eval)[1])

In [ ]:
np.linalg.norm(R_ddot_eval(*t_eval, *t_dot_eval, *t_ddot_eval) - jax.jvp(_get_R_dot_0, (t_eval,), (t_dot_eval,))[1] - jax.jvp(_get_R_dot_1, (t_dot_eval,), (t_ddot_eval,))[1])

## cost function visualization

In [ ]:
def leg_cost(ell):
    # asymmetrical, but better differentiation
    base = -np.log(ell**2 - min_leg_pos**2) - np.log(max_leg_pos**2 - ell**2)
    return base / 10.0

margin = (max_leg_pos - min_leg_pos) * 0.1
vals = np.linspace(min_leg_pos + margin, max_leg_pos - margin, 2**10)
fig, ax = plt.subplots()
ax.plot(vals, leg_cost(vals))
plt.show()

In [ ]:
def yaw_cost(yaw):
    # asymmetrical, but better differentiation
    base = 1.0 / (max_yaw**2 - yaw**2)
    return base / 10.0

margin = (2.0 * max_yaw) * 0.1
vals = np.linspace(-max_yaw + margin, max_yaw - margin, 2**10)
fig, ax = plt.subplots()
ax.plot(vals, yaw_cost(vals))
plt.show()

# messing around with casadi costs

In [ ]:
ell0 = ca.SX.sym("ell0")  # type: ignore
ell1 = ca.SX.sym("ell1")  # type: ignore
ell0, ell1

In [ ]:
cost = 0
leg_cost0 = (
    1.0 / (ell0**2 - min_leg_pos**2)**2 + 1.0 / (ell0**2 - max_leg_pos**2)**2
)
leg_cost0 /= 50.0
cost += leg_cost0
leg_cost1 = (
    1.0 / (ell1**2 - min_leg_pos**2)**2 + 1.0 / (ell1**2 - max_leg_pos**2)**2
)
leg_cost1 /= 50.0
cost += leg_cost1
assert type(cost) is ca.SX
type(cost)

## cartesian discrete integration?

In [ ]:
10.0 / dt

In [ ]:
x = 0
v = 0
acc = 1.0
for _ in range(2000):
    v = v + acc * dt
    x = x + v * dt
print(x)